In [ ]:
²from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# utilise le fichier csv pour l'AED

import pandas as pd
import ast # Import the ast module to safely evaluate the string representations of lists


csv_file_path = '/content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/input/Flipkart/flipkart_com-ecommerce_sample_1050.csv'

try:
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(csv_file_path)
    df['categories'] = df['product_category_tree'].apply(lambda x: [c.strip("[]'\"").strip() for c in x.split('>>')])
    df['top_level_category'] = df['product_category_tree'].apply(lambda x: ast.literal_eval(x)[0] if pd.notna(x) else None)

    print("\nAnalyse Exploratoire de Données (AED) du DataFrame:")
    print("\nAperçu des 5 premières lignes:")
    display(df.head())


except FileNotFoundError:
    print(f"Erreur: Le fichier CSV n'a pas été trouvé à l'adresse spécifiée: {csv_file_path}")
except Exception as e:
    print(f"Une erreur s'est produite lors de la lecture du fichier CSV: {e}")

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 1. Load images and labels
image_folder = os.path.join('/content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/input/Flipkart/', 'Images')
image_files = os.listdir(image_folder)

# Create a mapping from image filename to top-level category
df['image_filename_base'] = df['image'].apply(lambda x: os.path.basename(str(x)) if pd.notna(x) else None)
image_to_category = df.set_index('image_filename_base')['top_level_category'].to_dict()

images = []
labels = []
target_size = (224, 224)

for image_file in image_files:
    if image_file in image_to_category and image_to_category[image_file] is not None: # Add check for None category
        img_path = os.path.join(image_folder, image_file)
        try:
            with Image.open(img_path) as img:
                # Convert all images to RGB to handle grayscale images
                img = img.convert('RGB').resize(target_size)
                images.append(np.array(img))
                labels.append(image_to_category[image_file])
        except Exception as e:
            print(f"Could not process image {image_file}: {e}")

# Convert lists to numpy arrays
images = np.array(images)
labels = np.array(labels)

# Check class distribution before splitting
unique_labels, counts = np.unique(labels, return_counts=True)
classes_with_one_sample = unique_labels[counts == 1]
print(f"\nClasses with only one sample: {len(classes_with_one_sample)}")
# print(classes_with_one_sample) # Uncomment to see the actual labels

# 2. Preprocess images
images = images / 255.0  # Normalize to [0, 1]

# 3. Encode labels before splitting
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

onehot_encoder = OneHotEncoder(sparse_output=False)
labels_onehot = onehot_encoder.fit_transform(labels_encoded.reshape(-1, 1))


# 4. Split data
# Removed stratify due to classes with only one member
X_train, X_temp, y_train_onehot, y_temp_onehot = train_test_split(
    images, labels_onehot, test_size=0.3, random_state=42 # Removed stratify=labels
)

# Then, split temp into validation (15%) and testing (15%)
X_val, X_test, y_val_onehot, y_test_onehot = train_test_split(
    X_temp, y_temp_onehot, test_size=0.5, random_state=42 # Removed stratify=y_temp
)

print(f"\nTraining set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Testing set: {len(X_test)} samples")

print("\nShape of one-hot encoded training labels:", y_train_onehot.shape)
print("Shape of one-hot encoded validation labels:", y_val_onehot.shape)
print("Shape of one-hot encoded testing labels:", y_test_onehot.shape)

print("\nNumber of classes:", y_train_onehot.shape[1])

# Task
Perform a supervised classification on the images provided in the notebook.

## Data preparation

### Subtask:
Load, preprocess, and split the image data for supervised classification.


**Reasoning**:
The first step is to load the images and their labels, preprocess them by resizing and normalizing, and then split the data into training, validation, and testing sets, and finally one-hot encode the labels. I will write a single code block to perform all these steps as they are all related to data preparation for a supervised classification task.



## Model selection

### Subtask:
Choose a suitable model architecture for image classification. We can start with a simple CNN (Convolutional Neural Network) or use a pre-trained model for transfer learning (e.g., VGG16, ResNet, or EfficientNet).


**Reasoning**:
Define and compile a pre-trained VGG16 model for image classification.



## Model training

### Subtask:
Train the selected model on the training data.

**Reasoning**:
I will now train the compiled VGG16 model. I will use the `.fit()` method, providing the training and validation data. I'll set the number of epochs to 10 and the batch size to 32, and store the training history in a variable for later use.



In [ ]:
history = model.fit(
    X_train, y_train_onehot,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val_onehot)
)

In [ ]:
from tensorflow.keras.models import load_model

# Define the path to the saved model
model_load_path = '/content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/output//image_classification_model_augmented.h5'

# Load the model
model = load_model(model_load_path)

print(f"Model loaded successfully from: {model_load_path}")

In [ ]:
# Evaluate the model on the test set
print("\nEvaluating the model on the test set:")
loss, accuracy = model.evaluate(X_test, y_test_onehot, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Plot training and validation loss
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Assuming you have a new image, let's use an image from the test set as an example
# Replace this with your actual new image loading and preprocessing
# For a real-world scenario, you would load a new image, resize it to target_size (224, 224),
# convert it to RGB, and normalize it by dividing by 255.0

# Select an image from the test set
sample_image = X_test[0]
sample_label_onehot = y_test_onehot[0]

# The model expects a batch of images, so we need to add a batch dimension
sample_image_batch = np.expand_dims(sample_image, axis=0)

# Predict the class probabilities for the sample image
predictions = model.predict(sample_image_batch)

# Get the index of the class with the highest probability
predicted_class_index = np.argmax(predictions, axis=1)[0]

# Convert the predicted class index back to the original category label
predicted_category = label_encoder.inverse_transform([predicted_class_index])[0]

# Get the true category label (for comparison)
true_class_index = np.argmax(sample_label_onehot)
true_category = label_encoder.inverse_transform([true_class_index])[0]


print(f"Predicted Category: {predicted_category}")
print(f"True Category: {true_category}") # This is for comparison since we are using a test image

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

# Assuming you have loaded and preprocessed your images into a numpy array called 'images'
# and that X_train contains your training images.
# Let's pick a sample image from your training data to demonstrate augmentation.

if 'X_train' in locals() and len(X_train) > 0:
    sample_image = X_train[0] # Take the first image from the training set

    # Define the data augmentation layers
    data_augmentation_layers = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.2),
        # Add more augmentation layers as needed, e.g., RandomContrast, RandomTranslation, etc.
        # tf.keras.layers.RandomContrast(0.2),
        # tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    ])

    # Apply augmentation multiple times to see the effects
    num_augmentations = 5
    augmented_images = [sample_image] # Include the original image

    for _ in range(num_augmentations):
        # Data augmentation layers expect a batch, so add a batch dimension
        augmented_img = data_augmentation_layers(tf.expand_dims(sample_image, axis=0))
        # Remove the batch dimension to plot
        augmented_images.append(tf.squeeze(augmented_img, axis=0).numpy())

    # Plot the original and augmented images
    plt.figure(figsize=(15, 5))
    for i in range(num_augmentations + 1):
        plt.subplot(1, num_augmentations + 1, i + 1)
        plt.imshow(augmented_images[i])
        plt.title("Original" if i == 0 else f"Augmented {i}")
        plt.axis('off')
    plt.show()

else:
    print("X_train is not available. Please ensure you have loaded and preprocessed your image data.")

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Sequential

# Create a data augmentation model
data_augmentation = Sequential([
    RandomFlip("horizontal_and_vertical"),
    RandomRotation(0.2),
    RandomZoom(0.2),
], name="data_augmentation")

# You can apply this to your training data before feeding it to the model
# For example, you can create a new dataset that includes augmented images:
# augmented_X_train = data_augmentation(X_train)

# Alternatively, you can include the data augmentation layers directly in your model
# after the input layer. Let's modify your existing model to include this.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, Input

# Determine the number of classes from the one-hot encoded labels
num_classes = y_train_onehot.shape[1]

# Define the input layer
input_tensor = Input(shape=X_train.shape[1:])

# Add data augmentation layers
x = RandomFlip("horizontal_and_vertical")(input_tensor)
x = RandomRotation(0.2)(x)
x = RandomZoom(0.2)(x)

# Load the pre-trained VGG16 model, excluding the top classification layer
# Use include_top=False to remove the fully connected layers at the top
# Specify the input shape to match our image data (height, width, channels)
# The input_shape should match the shape of a single image sample from X_train
base_model = VGG16(weights='imagenet', include_top=False, input_tensor=x) # Connect VGG16 to the augmented output

# Freeze the weights of the pre-trained layers to prevent them from being updated during training
# This is a common practice in transfer learning, especially when the new dataset is small
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification layers on top of the pre-trained base
x = Flatten()(base_model.output) # Flatten the output of the last convolutional block
x = Dense(256, activation='relu')(x) # Add a dense layer with ReLU activation
predictions = Dense(num_classes, activation='softmax')(x) # Add the final output layer with softmax activation for classification

# Create the full model
model_augmented = Model(inputs=input_tensor, outputs=predictions)

# Compile the model
# Use Adam optimizer, CategoricalCrossentropy loss for multi-class classification, and accuracy metric
model_augmented.compile(optimizer=Adam(),
                        loss=CategoricalCrossentropy(),
                        metrics=['accuracy'])

# Display the model summary
print("\nModel Architecture Summary (with Data Augmentation):")
model_augmented.summary()

In [ ]:
# Train the augmented model
history_augmented = model_augmented.fit(
    X_train, y_train_onehot,
    epochs=10, # You can adjust the number of epochs
    batch_size=32,
    validation_data=(X_val, y_val_onehot)
)